# MonDecay on a free Kaggle/Colab T4
Turn on a GPU (T4) and internet. Re-running this notebook in a later session continues where it stopped: finished steps are skipped, interrupted ones resume.
**Untested on real GPU hardware:** read `docs/DECISIONS.md` ('Untested paths') first.

In [ ]:
!nvidia-smi | head -15
!git clone https://github.com/sykeriin/MonDecay.git
%cd MonDecay

In [ ]:
# torch is preinstalled on Kaggle/Colab; install the rest (no vLLM: T4 + free tier)
!pip install -q -e . peft trl datasets accelerate bitsandbytes

## Restore previous sessions' work
Kaggle: attach last session's output as a dataset and set `PERSIST` to `<that input path>/persist`. Colab: mount Drive and point at a Drive folder.

In [ ]:
PERSIST = '/kaggle/working/persist'   # change for Colab/Drive

In [ ]:
!python scripts/gpu/make_commands.py --profile kaggle-t4 > kaggle.txt
!cat kaggle.txt | cut -c1-160

## Run the queue (positive control first)
The budget leaves headroom for the platform's session limit. Look at the base run's gate before trusting anything after it.

In [ ]:
!python scripts/kaggle/run_queue.py kaggle.txt --persist {PERSIST} --budget-hours 8.5

## Gate on the base run, then verdicts
If the gate says FAIL, stop: the intervention results would not be interpretable.

In [ ]:
!mondecay loadbearing kaggle-t4-base deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B --suite data/suite/kaggle-t4
!mondecay gate kaggle-t4-base --suite data/suite/kaggle-t4

In [ ]:
!mondecay verdicts kaggle-t4-base qwen7b --suite data/suite/kaggle-t4
!mondecay leaderboard
!cat reports/practitioner_table.md